# Eagle5 v2 Corpus Build — Colab GPU

Builds the DeepSeek-V2-Lite-Chat calibration corpus for Eagle5 v2 draft-head training. The output (~94 parquet shards, ~1.5 GB total at int8 quantize) lands on **your Google Drive** so it survives Colab session disconnects.

## How to use

1. Open this notebook in Colab: `File → Upload notebook → eagle5_v2_corpus.ipynb`.
2. Set the runtime to GPU: `Runtime → Change runtime type → T4 GPU` (free) or `V100/A100` (Pro).
3. Run cells 1–7 in order. Watch for the `[overnight] GPU=...` print to confirm you got a GPU.
4. Cell 5 is the JavaScript keepalive (non-blocking — runs in the browser). Cell 6 is the corpus build (~5-6 hr on T4, ~2 hr on V100, ~1 hr on A100). Resumable: shards are written atomically, so a session disconnect just resumes from the last completed shard on the next run.
5. **Single-tab workflow:** Run Cell 5 first (JS keepalive, non-blocking) then Cell 6 (corpus build). The JS persists in the browser regardless of what Python is doing — no second tab needed.
6. After completion (Cell 7 verifies), the corpus is on Drive at `MyDrive/dismantle/v2_lite_corpus/`. Download to laptop and run training locally with MLX.

## Hardware fit & auto-config

DeepSeek-V2-Lite is 16B params (~32 GB fp16). Cell 6 auto-detects GPU and picks the best config:

| GPU | VRAM | Strategy | Corpus seqs | Batch | ETA |
|---|---|---|---|---|---|
| **Blackwell** (Pro+) | 102 GB | Native fp16 | **100,000** | 32 | ~30 min |
| **A100 80GB** (Pro+) | 80 GB | Native fp16 | **100,000** | 16 | ~1 hr |
| **H100** (Pro+) | 80 GB | Native fp16 | **100,000** | 32 | ~1 hr |
| **A100 40GB** (Pro/Pro+) | 40 GB | Native fp16 | **50,000** | 12 | ~1 hr |
| **L4** (Pro) | 24 GB | 4-bit nf4 | **30,000** | 8 | ~1.5 hr |
| **V100** (Pro) | 16 GB | 4-bit + CPU offload | 20,000 | 4 | ~2 hr |
| **T4** (Free/Pro) | 16 GB | 4-bit + CPU offload | 20,000 | 4 | ~2.5 hr |

**To request H100:** `Runtime → Change runtime type → Hardware accelerator → A100 GPU → then in the dropdown select H100`. If H100 unavailable, fall back to A100 → L4 → V100 → T4. Notebook auto-adjusts.

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch, subprocess
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime → Change runtime type → GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[overnight] GPU={gpu_name}  VRAM={vram_gb:.1f} GB")
print(subprocess.check_output(['nvidia-smi']).decode())

In [ ]:
# Cell 2 — Install deps (CRITICAL: pin transformers to v4.x for V2-Lite)
#
# transformers v5 broke DeepSeek-V2-Lite MoE loading. The checkpoint
# stores fused per-MoE-block tensors (e.g., `experts.down_proj` per
# layer) but the modeling code expects per-expert split tensors
# (`experts.{0..63}.down_proj.weight`). transformers v5's loader
# doesn't convert between these formats, so the per-expert weights
# end up RANDOMLY INITIALIZED and the model produces garbage.
#
# transformers v4.45-v4.50 handles V2-Lite cleanly. Downgrade Colab's
# default v5 install. AFTER THIS CELL: RUNTIME RESTART REQUIRED.

import sys
needs_restart = "transformers" in sys.modules

!pip install -q --upgrade \
  "transformers>=4.45,<5" \
  "bitsandbytes>=0.43" \
  "accelerate>=1.0" \
  "tokenizers<0.20"

if needs_restart:
    print()
    print("=" * 60)
    print("⚠️  RESTART RUNTIME NOW")
    print("    Runtime → Restart runtime")
    print("    Then re-run from Cell 1 (Mount Drive)")
    print("=" * 60)
else:
    import transformers
    if not transformers.__version__.startswith("4."):
        print(f"⚠️  Got transformers {transformers.__version__} — expected 4.x")
        print("    Runtime → Restart runtime; re-run from Cell 1")
    else:
        print(f"✅ transformers {transformers.__version__} (correct, v4 — V2-Lite loads cleanly)")

In [ ]:
# Cell 3 — Clone dismantle repo
import os
if not os.path.exists('/content/dismantle'):
    !cd /content && git clone --depth 1 https://github.com/joshuahickscorp/dismantle.git
else:
    !cd /content/dismantle && git pull --ff-only
%cd /content/dismantle
!ls tools/training/build_corpus.py

In [ ]:
# Cell 4 — Patch HF cache for transformers v5 compatibility (NO-OP on v4)
#
# If Cell 2 pinned transformers to v4 (the default), this cell does nothing
# because v4 already has is_torch_fx_available and the modeling file imports
# cleanly. Keep it for backward-compat — runs in ~2 sec, does no harm.
#
# DeepSeek-V2-Lite-Chat's modeling_deepseek.py imports `is_torch_fx_available`
# from transformers.utils.import_utils, which was removed in transformers v5.
# Colab ships transformers v5.x, so the load fails. We trigger an
# AutoModelForCausalLM load (which downloads the modeling .py before trying
# to import it), catch the expected ImportError, then patch the cached file
# so the next load succeeds.
#
# Must run BEFORE Cell 4 (the corpus build) in every fresh Colab session.
# Idempotent — re-running is safe.

import glob
from transformers import AutoModelForCausalLM
import transformers
print(f"transformers {transformers.__version__}")

try:
    AutoModelForCausalLM.from_pretrained(
        "deepseek-ai/DeepSeek-V2-Lite-Chat",
        trust_remote_code=True,
    )
    print("(model load unexpectedly succeeded — already patched?)")
except ImportError as e:
    print(f"(triggered cache via expected ImportError: {str(e)[:80]}…)")
except Exception as e:
    print(f"(load errored with {type(e).__name__}; checking if file cached anyway)")

candidates = glob.glob(
    "/root/.cache/huggingface/modules/transformers_modules/**/modeling_deepseek.py",
    recursive=True)
assert candidates, "modeling_deepseek.py STILL not in HF cache — investigate with `!find /root/.cache/huggingface -name modeling_deepseek.py`"
target = candidates[0]
print(f"patching: {target}")

src = open(target).read()
needle = "from transformers.utils.import_utils import is_torch_fx_available"
if needle not in src:
    print("⚠️  needle not found — already patched or upstream changed (likely safe)")
else:
    patched = src.replace(needle, """try:
    from transformers.utils.import_utils import is_torch_fx_available
except ImportError:
    def is_torch_fx_available():
        return False""")
    open(target, "w").write(patched)
    print("✅ patched — Cell 5 will now load past the import error")

In [ ]:
# Cell 5 — START KEEPALIVE (non-blocking, single-tab)
#
# Injects a JavaScript snippet that simulates user activity every 60 sec by
# clicking the Colab connect button. JS runs in the BROWSER, not the Python
# kernel, so Cell 6 (corpus build) runs in parallel — no second tab needed.
#
# Run this ONCE before Cell 6. It persists for the lifetime of the browser
# tab (until you close or reload it). Output in browser DevTools console
# (F12 → Console) shows ticks if you want to verify.

from IPython.display import display, Javascript
display(Javascript("""
function _dismantle_keepalive() {
  try {
    const btn = document.querySelector("colab-connect-button");
    if (btn && btn.shadowRoot) {
      const inner = btn.shadowRoot.querySelector("#connect");
      if (inner) inner.click();
    }
    document.dispatchEvent(new KeyboardEvent("keydown", {key: "Shift"}));
    console.log("[dismantle keepalive] tick " + new Date().toISOString());
  } catch (e) {
    console.warn("[dismantle keepalive] error", e);
  }
}
_dismantle_keepalive();                       // initial tick
setInterval(_dismantle_keepalive, 60 * 1000); // every 60 sec
"""))
print("✅ keepalive injected — clicks connect-button every 60 sec via JS")
print("   Non-blocking. Proceed to Cell 6 in this same tab.")
print("   (F12 → Console to see ticks if curious; reload page to stop)")

In [ ]:
# Cell 6 — RUN CORPUS BUILD (auto-tunes for whichever GPU Colab gave you)
#
# Per-GPU configs (auto-detected from VRAM):
#   102 GB (Blackwell):  fp16, batch=32, 100,000 seqs (~30 min)
#   80 GB  (A100 80GB):  fp16, batch=16, 100,000 seqs (~1 hr)
#   40 GB  (A100 40GB):  fp16, batch=12,  50,000 seqs (~1 hr)
#   24 GB  (L4):         4-bit nf4, batch=8, 30,000 seqs (~1.5 hr)
#   16 GB  (V100/T4):    4-bit + CPU offload, batch=4, 20k seqs (~2 hr)
#
# CRITICAL: passes --no-trust-remote-code so transformers uses its NATIVE
# DeepSeek-V2 loader instead of the broken remote modeling_deepseek.py
# (the remote file expects per-expert split tensors but the checkpoint
# has fused per-block tensors → experts initialize as RANDOM = garbage).

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # alias on some versions

OUT = "/content/drive/MyDrive/dismantle/v2_lite_corpus"
os.makedirs(OUT, exist_ok=True)
existing = len([f for f in os.listdir(OUT) if f.endswith(".parquet")])

import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)
print(f"[overnight] GPU={gpu_name}  VRAM={vram_gb:.1f} GB")

if vram_gb >= 100:
    # Blackwell 102 GB — keep batch modest so first-batch CUDA warmup is
    # bearable (batch=32 took 17 sec for the first iter; batch=16 is ~6 sec
    # and still saturates Blackwell compute since it's bandwidth-limited).
    tier = "Blackwell"
    seqs = 100000
    batch = 16
    load_flag = ""
    vram_cap_flag = ""
elif vram_gb >= 70:
    # A100 80GB — fits V2-Lite native fp16 (32GB) but batch=32 OOMs at
    # 4096 tokens. batch=16 leaves ~12 GB activations headroom.
    tier = "A100-80GB"
    seqs = 100000
    batch = 16
    load_flag = ""
    vram_cap_flag = ""
elif vram_gb >= 35:
    # A100 40GB — fp16 model 32 GB + batch=12 activations ~6 GB = 38 GB
    tier = "A100-40GB"
    seqs = 50000
    batch = 12
    load_flag = ""
    vram_cap_flag = ""
elif vram_gb >= 20:
    # L4 24GB — needs 4-bit (fp16 32GB doesn't fit), no offload
    tier = "L4"
    seqs = 30000
    batch = 8
    load_flag = "--load-4bit"
    vram_cap_flag = ""
else:
    # T4/V100 16GB — 4-bit + CPU offload (smallest config)
    tier = "T4/V100"
    seqs = 20000
    batch = 4
    load_flag = "--load-4bit"
    vram_cap_flag = "--cuda-max-vram-gb 12"

target_shards = seqs // 32
print(f"[overnight] tier={tier}  seqs={seqs}  batch={batch}  "
      f"max-tokens=4096  target_shards={target_shards}")
print(f"[overnight] existing on Drive: {existing} / {target_shards}")
print(f"[overnight] load={load_flag or 'native fp16'}  cap={vram_cap_flag or 'unrestricted'}")
print(f"[overnight] trust_remote_code=False (native HF DeepSeek-V2 loader)")

# Sanity check: transformers version
import transformers
tv = transformers.__version__
print(f"[overnight] transformers={tv}")
if not tv.startswith("4."):
    print(f"(transformers {tv} — native loader path handles V2-Lite OK)")

!PYTORCH_ALLOC_CONF=expandable_segments:True \
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python tools/training/build_corpus.py \
  --model deepseek-ai/DeepSeek-V2-Lite-Chat \
  --dataset HuggingFaceH4/ultrachat_200k \
  --max-sequences {seqs} \
  --batch-size {batch} \
  --max-tokens-per-seq 4096 \
  --shard-size 32 \
  --device cuda \
  --dtype float16 \
  --capture all \
  --no-trust-remote-code \
  {load_flag} \
  {vram_cap_flag} \
  --out {OUT}

In [ ]:
# Cell 7 — Verify + summarize when corpus is done
import os, glob
OUT = '/content/drive/MyDrive/dismantle/v2_lite_corpus'
shards = sorted(glob.glob(f'{OUT}/*.parquet'))
if not shards:
    print(f"⚠️  no shards at {OUT} — corpus build hasn't completed")
else:
    total = sum(os.path.getsize(s) for s in shards) / 1e6
    print(f"✅ {len(shards)} shards on Drive, total {total:.1f} MB")
    print(f"   first: {os.path.basename(shards[0])}")
    print(f"   last:  {os.path.basename(shards[-1])}")
    print()
    print("Download to laptop:")
    print(f"   1. Open Drive: drive.google.com → MyDrive/dismantle/v2_lite_corpus/")
    print(f"   2. Right-click the folder → Download (Drive zips it)")
    print(f"   3. On laptop:  unzip → mv to ~/Downloads/dismantle/artifacts/calibration/v2_lite_corpus/")
    print(f"   4. Run laptop chain:  tools/bench/overnight_eagle5_2026_05_26.sh")
    print(f"      (chain detects existing corpus and skips straight to training)")

In [ ]:
# Cell 8 (OPTIONAL) — Run training on Colab too
#
# If you'd rather not download the corpus + train on laptop, you can train
# right here on Colab. Caveat: eagle5_train.py uses MLX which is Apple-only;
# this cell ports the training to PyTorch on the fly. ~30 min on T4.
#
# Generally not worth it — MLX training on laptop is faster than PyTorch on
# T4 once the corpus is local. Skip this cell unless you need the head
# without downloading the corpus.

print("Skipping — train on laptop with MLX. See README in colab/ folder.")